# Final Feature Engineering for 24-Hour Air Quality Forecasting

This notebook focuses on generating a common feature pool for both Tree-based and Deep Learning models. 

**Generated Features:**
- **Temporal & HCMC-Specific Seasons**: Rush hours, Dry/Rainy seasons, Cyclic time parts.
- **Extended Lags & Rolling Stats**: Capturing dependencies up to 72 hours back.
- **Momentum & Trends**: Rate of change in pollution and weather patterns.
- **Meteorological Physics**: Washing effect of rain, Wind U/V vectors, and Wind categories.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_context('notebook')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Data

In [2]:
df = pd.read_csv('../data/processed/hcmc_merged_cleaned.csv')
df['datetime_local'] = pd.to_datetime(df['datetime_local'])
df = df.set_index('datetime_local')
df = df.sort_index()

print(f"Loaded {len(df)} rows and {len(df.columns)} columns.")
df.head()

Loaded 10735 rows and 12 columns.


,pm1,pm25,relativehumidity,temperature,um003,temperature_2m,relative_humidity_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m,boundary_layer_height
datetime_local,,,,,,,,,,,,
2024-11-19 18:00:00+07:00,18.600000,29.139999,62.800000,27.172000,3519.660059,30.9,59.0,0.1,1007.6,1.0,211.0,150.0
2024-11-19 19:00:00+07:00,18.700000,29.150000,57.816667,29.025001,3363.499980,29.3,68.0,0.0,1008.4,4.9,204.0,245.0
2024-11-19 20:00:00+07:00,20.533333,31.783333,57.333333,29.130000,3903.650004,26.6,77.0,0.2,1009.5,2.8,105.0,95.0
2024-11-19 21:00:00+07:00,20.016667,30.950000,56.883334,29.161667,3583.299988,26.6,79.0,0.1,1010.0,4.0,309.0,85.0
2024-11-19 22:00:00+07:00,19.679167,30.216667,56.395833,29.043750,3495.024984,26.8,79.0,0.0,1009.9,3.2,164.0,55.0


## 2. Temporal & HCMC-Specific Feature Engineering

In [3]:
def extract_time_features(df):
    df = df.copy()
    df['hour'] = df.index.hour
    df['day_of_week'] = df.index.dayofweek
    df['month'] = df.index.month
    
    # Weekend feature
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    
    # Rush Hour peaks (HCMC: 7-9 AM and 5-7 PM)
    df['is_rush_hour'] = df['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
    
    # HCMC Seasons (Dry: Nov-Apr, Rainy: May-Oct)
    df['is_dry_season'] = df['month'].isin([11, 12, 1, 2, 3, 4]).astype(int)
    
    # Cyclic encoding for Hour (0-23)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    
    # Cyclic encoding for Month (1-12)
    df['month_sin'] = np.sin(2 * np.pi * (df['month']-1) / 12)
    df['month_cos'] = np.cos(2 * np.pi * (df['month']-1) / 12)
    
    return df

df = extract_time_features(df)

## 3. Extended Lags and Multi-scale Rolling Statistics

In [4]:
def create_lags_and_rolling(df, target='pm25', lags=[1, 2, 3, 6, 12, 24, 48, 72], windows=[6, 12, 24, 48, 72]):
    df_eng = df.copy()
    for lag in lags:
        df_eng[f'{target}_lag_{lag}'] = df_eng[target].shift(lag)
    for window in windows:
        df_eng[f'{target}_roll_{window}h_mean'] = df_eng[target].rolling(window=window).mean()
        df_eng[f'{target}_roll_{window}h_std'] = df_eng[target].rolling(window=window).std()
    return df_eng

df = create_lags_and_rolling(df)

## 4. Meteorological Physics & Domain Features

In [5]:
def add_meteorological_physics(df):
    df = df.copy()
    
    # 1. Washing Effect: Precipitation logic
    df['is_raining'] = (df['precipitation'] > 0).astype(int)
    
    # Counter for hours since last rain
    rain_mask = df['is_raining'] == 1
    df['hours_since_last_rain'] = df.groupby(rain_mask.cumsum()).cumcount()
    df.loc[df['is_raining'] == 1, 'hours_since_last_rain'] = 0
    
    # 2. Wind Vectorization (U and V components)
    wind_rad = np.radians(df['wind_direction_10m'])
    df['wind_u'] = df['wind_speed_10m'] * np.cos(wind_rad)
    df['wind_v'] = df['wind_speed_10m'] * np.sin(wind_rad)
    
    # 3. Wind Condition
    def categorize_wind(speed):
        if speed < 0.5: return 'Calm'
        if speed < 2.0: return 'Light'
        return 'Moderate+'
    
    df['wind_condition'] = df['wind_speed_10m'].apply(categorize_wind)
    
    # 4. Momentum / Trends
    df['pm25_diff_1h'] = df['pm25'].diff(1)
    df['pm25_diff_24h'] = df['pm25'].diff(24)
    df['pressure_trend_3h'] = df['surface_pressure'].diff(3)
    
    return df

df = add_meteorological_physics(df)

## 5. Cleaning & Final Export

In [6]:
initial_count = len(df)
df_final = df.dropna()
final_count = len(df_final)

print(f"Initial data points: {initial_count}")
print(f"Final data points: {final_count} (Dropped {initial_count - final_count} rows due to lag history)")

df_final.to_csv('../data/processed/hcmc_features.csv')
print("Saved specialized features to data/processed/hcmc_features.csv")

Initial data points: 10735
Final data points: 10663 (Dropped 72 rows due to lag history)


Saved specialized features to data/processed/hcmc_features.csv


## 6. Feature Dictionary

| Feature Category | Name | Meaning / Physical Importance |
| :--- | :--- | :--- |
| **Temporal** | `hour`, `month` | Daily residential patterns and seasonal cycles. |
| | `is_rush_hour` | High traffic emission periods (7-9 AM, 5-7 PM). |
| | `is_dry_season` | High pollution period (Nov-Apr) vs washout period. |
| **Autoregressive** | `pm25_lag_1` to `72` | Historical pollution levels (1 hour to 3 days back). |
| **Trend/Rolling** | `pm25_diff_24h` | Difference between current pollution and same time yesterday. |
| **Rain Physics** | `hours_since_last_rain`| Accumulation time since the last total washout event. |
| **Wind Physics** | `wind_u`, `wind_v` | Vector components for continuous direction handling. |